<a href="https://colab.research.google.com/github/baokhanh546123/Build-Knowledge-Graph-Simple/blob/main/Fix%2BKG_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q pyspark==3.5.6 spark-nlp==5.5.0 graphframes-py==0.12.3 sentence-transformers  networkx matplotlib psutil datasets transformers

In [2]:
from __future__ import annotations
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import (
    SentenceDetector, Tokenizer, WordEmbeddingsModel,
    NerDLModel, NerConverter, PerceptronModel,
    DependencyParserModel, TypedDependencyParserModel
)
from pyspark.ml import Pipeline
from pyspark.sql import functions as F , SparkSession, DataFrame
from pyspark.sql.types import StringType
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from dataclasses import dataclass
from typing import Tuple, Dict, List, Optional
from datasets import load_dataset

import sparknlp
import numpy as np
import pandas as pd
import gc , os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages io.graphframes:graphframes-spark3_2.12:0.12.3,"
    "com.johnsnowlabs.nlp:spark-nlp_2.12:5.5.0 "
    "pyspark-shell"
)

from graphframes import GraphFrame

In [3]:
@dataclass
class Config:
  OUTPUT_DIR : str = "kg_output"
  PARTITIONS : int = 2
  AUTOBROADCASTJOINTHRESHOLD : float = -1
  DRIVER_MEMORY = "2g"
  ARROW_MOD : bool = True

  def __post_init__(self):
    if not os.path.exists(self.OUTPUT_DIR):
      os.makedirs(self.OUTPUT_DIR , exist_ok=True)

@dataclass
class Model_Embedding:
  EMBEDDING_MODEL : str = "BAAI/bge-small-en-v1.5"
  SIMILARITY_THRESHOLD : float = 0.84
  BATCH_SIZE : int = 32
  show_progress_bar : bool = False
  convert_to_numpy : bool = True


In [4]:
import json

def normalize_wikipedia_record(record):
    document_id = str(record.get("id", "")).strip()
    url = record.get("url", "") or ""
    title = record.get("title", "") or ""
    content = record.get("text", "") or ""

    # Bỏ qua record không có nội dung
    if not content.strip():
        return None

    metadata = {
        "dataset": "wikimedia/wikipedia",
        "version": "20231101.en",
        "language": "en"
    }

    return {
        "document_id": document_id,
        "metadata": json.dumps(metadata, ensure_ascii=False),
        "source_link": url,
        "title": title,
        "content": content.strip()
    }

In [5]:
from datasets import load_dataset

df = load_dataset(
    "wikimedia/wikipedia",
    data_dir="20231101.en",
    token=True,
    streaming=True
)

train = df["train"]

train_shuffle = train.shuffle(
    seed=42,
    buffer_size=1000
)

random_sample = train_shuffle.take(1000)

# Chuẩn hóa từng record
sample_list = []

for record in random_sample:
    normalized = normalize_wikipedia_record(record)

    if normalized is not None:
        sample_list.append(normalized)

print(f"Total documents: {len(sample_list)}")
print(sample_list[:2])

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Total documents: 1000
[{'document_id': '64505424', 'metadata': '{"dataset": "wikimedia/wikipedia", "version": "20231101.en", "language": "en"}', 'source_link': 'https://en.wikipedia.org/wiki/Tom%C3%A1s%20Reym%C3%A3o', 'title': 'Tomás Reymão', 'content': "Tomás Mascarenhas Lemos Reymão Nogueira (born 14 July 1998) is a Portuguese professional footballer who plays as a midfielder for Liga Portugal 2 club Leixões. As a youth player, he spent time at clubs such as Sporting CP, Chelsea, Wolverhampton Wanderers and Fiorentina.\n\nClub career\nReymão made his professional debut with Boavista F.C. in a 1-0 Primeira Liga loss to C.S. Marítimo on 8 July 2020. In the axadrezada team, despite his youth and the change of coaches, he has been consolidating his position in the squad, now in Petit's options, and he is one of the current great hopes of the Black Panthers' supporters.\n\nOn 23 July 2022, Reymão moved abroad and joined Spanish Segunda División side Albacete Balompié on a one-year contrac

In [6]:
def setup_spark(config : Config) -> SparkSession:
  spark = sparknlp.start()
  spark.conf.set("spark.sql.shuffle.partitions", config.PARTITIONS)
  spark.conf.set("spark.sql.autoBroadcastJoinThreshold", config.AUTOBROADCASTJOINTHRESHOLD)
  #spark.conf.set("spark.driver.memory", "2g")
  spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", config.ARROW_MOD)
  return spark

In [7]:
def build_nlp_pipeline() -> Pipeline:
    document_assembler = DocumentAssembler() \
        .setInputCol("content") \
        .setOutputCol("document")

    sentence_detector = SentenceDetector() \
        .setInputCols(["document"]) \
        .setOutputCol("sentence")

    tokenizer = Tokenizer() \
        .setInputCols(["sentence"]) \
        .setOutputCol("token")

    embeddings = WordEmbeddingsModel.pretrained("glove_100d", "en") \
        .setInputCols(["sentence", "token"]) \
        .setOutputCol("embeddings")

    ner_tagger = NerDLModel.pretrained("ner_dl", "en") \
        .setInputCols(["sentence", "token", "embeddings"]) \
        .setOutputCol("ner")

    ner_converter = NerConverter() \
        .setInputCols(["sentence", "token", "ner"]) \
        .setOutputCol("ner_chunk")

    return Pipeline(stages=[
        document_assembler,
        sentence_detector,
        tokenizer,
        embeddings,
        ner_tagger,
        ner_converter,
    ])

In [8]:
def run_pipeline(spark: SparkSession, input_df: DataFrame) -> DataFrame:
    # Build the Spark NLP pipeline
    pipeline = build_nlp_pipeline()

    # Run the pipeline directly on the PySpark input DataFrame
    result_df = (
        pipeline.fit(input_df)
        .transform(input_df)
    )

    return result_df

In [9]:
def normalize_entity_name(column):
    return F.lower(
        F.trim(
            F.regexp_replace(
                column,
                r"\s+",
                " "
            )
        )
    )

In [10]:
def extract_graph_data(
    result_df: DataFrame
) -> Tuple[DataFrame, DataFrame, DataFrame]:

    # ==========================================
    # 1. Extract entity occurrences
    # ==========================================

    occurrences = (
        result_df
        .select(
            "document_id",
            F.explode("ner_chunk").alias("chunk")
        )
        .select(
            F.col("chunk.result").alias("original"),
            F.col("chunk.metadata.entity").alias("entity_type"),
            F.col("chunk.metadata.sentence")
                .cast("int")
                .alias("sentence_id"),
            F.col("chunk.begin").alias("begin"),
            F.col("chunk.end").alias("end"),
            F.col("document_id")
        )
        .filter(
            F.col("original").isNotNull() &
            F.col("entity_type").isNotNull()
        )
        .withColumn(
            "normalized_name",
            normalize_entity_name(F.col("original"))
        )
        .withColumn(
            "id",
            F.concat_ws(
                "::",
                F.col("entity_type"),
                F.col("normalized_name")
            )
        )
        .filter(F.length("normalized_name") > 0)
        .dropDuplicates([
            "document_id",
            "sentence_id",
            "begin",
            "end",
            "id"
        ])
    )

    # ==========================================
    # 2. Build vertices
    # ==========================================

    vertices = (
        occurrences
        .groupBy("id", "entity_type")
        .agg(
            F.min("original").alias("name"),
            F.count("*").alias("frequency")
        )
        .select(
            "id",
            "name",
            "entity_type",
            "frequency"
        )
    )

    # ==========================================
    # 3. Generate candidate co-occurrences
    # ==========================================

    a = occurrences.alias("a")
    b = occurrences.alias("b")

    pair_candidates = (
        a.join(
            b,
            on=[
                F.col("a.document_id") ==
                F.col("b.document_id"),

                F.col("a.sentence_id") ==
                F.col("b.sentence_id")
            ],
            how="inner"
        )
        .filter(
            F.col("a.id") < F.col("b.id")
        )
        .select(
            F.col("a.id").alias("src"),
            F.col("b.id").alias("dst"),

            F.col("a.entity_type").alias("src_type"),
            F.col("b.entity_type").alias("dst_type"),

            F.col("a.document_id").alias("document_id"),
            F.col("a.sentence_id").alias("sentence_id")
        )
    )

    # ==========================================
    # 4. Aggregate edges
    # ==========================================

    edges = (
        pair_candidates
        .groupBy(
            "src",
            "dst",
            "src_type",
            "dst_type"
        )
        .agg(
            F.count("*").alias("pair_count"),

            F.countDistinct(
                "document_id",
                "sentence_id"
            ).alias("sentence_count"),

            F.collect_set(
                F.struct(
                    "document_id",
                    "sentence_id"
                )
            ).alias("evidence")
        )
        .withColumn(
            "relationship",
            F.lit("CO_OCCURS")
        )
        .select(
            "src",
            "dst",
            "relationship",
            "src_type",
            "dst_type",
            "pair_count",
            "sentence_count",
            "evidence"
        )
    )

    return vertices, edges, occurrences

In [11]:
def load_embedding_model(config : Model_Embedding) -> Optional[SentenceTransformer]:
    try:
        model_name = config.EMBEDDING_MODEL or "BAAI/bge-small-en-v1.5"

        print(f"[INFO] Loading model {model_name}")
        model = SentenceTransformer(model_name, device="cpu")
        return model
    except Exception as e:
        print(f"[Error] Cannot load model {model_name}: {e}")
        return None

In [12]:
def normalize_entities_with_embeddings(
    config : Model_Embedding,
    model: SentenceTransformer,
    entity_list: List[str]
) -> Dict[str, str]:

    threshold: float = config.SIMILARITY_THRESHOLD or .84

    if not entity_list:
        return {}

    embeddings = model.encode(
        entity_list,
        show_progress_bar=config.show_progress_bar,
        convert_to_numpy=config.convert_to_numpy,
        batch_size=config.BATCH_SIZE          # giảm batch nếu RAM thấp
    )

    sim_matrix = cosine_similarity(embeddings)
    n = len(entity_list)
    assigned = [-1] * n
    clusters = []

    for i in range(n):
        if assigned[i] != -1:
            continue
        cluster = [i]
        assigned[i] = len(clusters)
        for j in range(i + 1, n):
            if assigned[j] == -1 and sim_matrix[i, j] >= threshold:
                cluster.append(j)
                assigned[j] = len(clusters)
        clusters.append(cluster)

    mapping = {}
    for cluster in clusters:
        names = [entity_list[idx] for idx in cluster]
        # Ưu tiên tên dài hơn làm canonical
        canonical = max(names, key=lambda x: (len(x), x))
        for name in names:
            mapping[name] = canonical

    return mapping

In [13]:
def apply_normalization(
    config: Model_Embedding,
    spark: SparkSession,
    result_df: DataFrame,
    model: SentenceTransformer
) -> Tuple[DataFrame, DataFrame]:
    # Collect unique entities
    unique_entities = (
        result_df
        .select(F.explode("ner_chunk").alias("c"))
        .select(F.col("c.result").alias("entity"))
        .distinct()
        .rdd.flatMap(lambda x: x)
        .collect()
    )

    print(f"Unique entities before normalization: {len(unique_entities)}")

    entity_mapping = normalize_entities_with_embeddings(config, model, unique_entities)

    # Print merged entities
    merged = {k: v for k, v in entity_mapping.items() if k != v}
    if merged:
        print("\n=== Merged entities ===")
        for orig, canon in sorted(merged.items(), key=lambda x: x[1]):
            print(f"{orig:<50} → {canon}")

    broadcast_mapping = spark.sparkContext.broadcast(entity_mapping)

    def map_entity(entity):
        if entity is None:
            return None
        return broadcast_mapping.value.get(entity, entity)

    map_entity_ = F.udf(map_entity, StringType())

    normalized = (
        result_df.select(
            "document_id",
            F.explode("ner_chunk").alias("c")
        )
        .select(
            F.col("document_id"),
            F.col("c.result").alias("original"),
            map_entity_("c.result").alias("canonical"),
            F.col("c.metadata.entity").alias("entity_type"),
            F.col("c.metadata.sentence").cast("int").alias("sentence_id"),
            F.col("c.begin").alias("begin"),
        )
        .withColumn(
            "normalized_name",
            normalize_entity_name(F.col("canonical"))
        )
        .withColumn(
            "id",
            F.concat_ws(
                "::",
                F.col("entity_type"),
                F.col("normalized_name")
            )
        )
        .filter(F.length("normalized_name") > 0)
    )

    vertices_norm = (
        normalized
        .groupBy("id", "entity_type")
        .agg(
            F.min("canonical").alias("name"),
            F.count("*").alias("frequency")
        )
    )

    entities_norm = normalized.select(
        "document_id",
        "id",
        "entity_type",
        "sentence_id",
        "begin"
    )

    edges_norm = (
        entities_norm.alias("a")
        .join(
            entities_norm.alias("b"),
            on=[
                F.col("a.document_id") == F.col("b.document_id"),
                F.col("a.sentence_id") == F.col("b.sentence_id"),
            ],
            how="inner"
        )
        .filter(
            (F.col("a.id") < F.col("b.id"))
        )
        .select(
            F.col("a.id").alias("src"),
            F.col("b.id").alias("dst"),
            F.lit("CO_OCCURS").alias("relationship"),
            F.col("a.entity_type").alias("src_type"),
            F.col("b.entity_type").alias("dst_type")
        )
        .dropDuplicates(["src", "dst"])
    )

    # Free resources
    broadcast_mapping.unpersist()
    del entity_mapping, unique_entities
    gc.collect()

    return vertices_norm, edges_norm

In [14]:
def save_dataframe(config : Config ,  df: DataFrame, path: str, format: str = "parquet"):
    """Ghi file và giải phóng cache nếu có"""
    full_path = os.path.join(config.OUTPUT_DIR, path)

    if format == "parquet":
        df.write.mode("overwrite").parquet(full_path)
    elif format == "json":
        df.write.mode("overwrite").json(full_path)
    else:
        raise ValueError("Unsupported format")

    print(f"Saved → {full_path}")

In [15]:
import os
import time
import threading
import psutil


class ResourceMonitor:
    def __init__(self, spark, interval=0.2):
        self.spark = spark
        self.interval = interval

        self.process = psutil.Process(os.getpid())

        self.running = False
        self.thread = None

        self.rss_start_mb = 0.0
        self.rss_end_mb = 0.0
        self.peak_rss_mb = 0.0
        self.peak_python_mb = 0.0
        self.peak_java_mb = 0.0

        self.gc_before = {}
        self.gc_after = {}

    # Lấy thông tin JVM Garbage Collector
    def _get_jvm_gc(self):
        result = {}

        try:
            management = (
                self.spark.sparkContext._jvm
                .java.lang.management.ManagementFactory
            )

            gc_beans = management.getGarbageCollectorMXBeans()
            iterator = gc_beans.iterator()

            while iterator.hasNext():
                bean = iterator.next()

                name = str(bean.getName())
                collection_time = int(bean.getCollectionTime())
                collection_count = int(bean.getCollectionCount())

                result[name] = {
                    "time_ms": max(collection_time, 0),
                    "count": max(collection_count, 0)
                }

        except Exception as e:
            print(f"[ResourceMonitor] Cannot read JVM GC: {e}")

        return result

    # Tính RSS của một process và toàn bộ process con
    def _get_process_memory(self):
        python_rss = 0
        java_rss = 0
        total_rss = 0

        try:
            processes = [self.process]
            processes.extend(self.process.children(recursive=True))

            for proc in processes:
                try:
                    rss = proc.memory_info().rss

                    total_rss += rss

                    process_name = proc.name().lower()

                    if "java" in process_name:
                        java_rss += rss
                    elif proc.pid == self.process.pid:
                        python_rss += rss

                except (psutil.NoSuchProcess, psutil.AccessDenied):
                    continue

        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

        mb = 1024 * 1024

        return (
            total_rss / mb,
            python_rss / mb,
            java_rss / mb
        )

    # Lấy mẫu RSS định kỳ
    def _sampling_loop(self):
        while self.running:
            total, python_mem, java_mem = self._get_process_memory()

            self.peak_rss_mb = max(self.peak_rss_mb, total)
            self.peak_python_mb = max(self.peak_python_mb, python_mem)
            self.peak_java_mb = max(self.peak_java_mb, java_mem)

            time.sleep(self.interval)

    def start(self):
        # GC tại thời điểm bắt đầu
        self.gc_before = self._get_jvm_gc()

        # RSS tại thời điểm bắt đầu
        total, _, _ = self._get_process_memory()

        self.rss_start_mb = total
        self.peak_rss_mb = total

        self.running = True

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(self):
        self.running = False

        if self.thread is not None:
            self.thread.join()

        # RSS tại thời điểm kết thúc
        total, _, _ = self._get_process_memory()

        self.rss_end_mb = total
        self.peak_rss_mb = max(self.peak_rss_mb, total)

        # GC tại thời điểm kết thúc
        self.gc_after = self._get_jvm_gc()

        return self.report()

    def report(self):
        gc_time_ms = 0
        gc_count = 0

        gc_details = []

        all_gc_names = set(self.gc_before) | set(self.gc_after)

        for name in all_gc_names:
            before = self.gc_before.get(
                name,
                {"time_ms": 0, "count": 0}
            )

            after = self.gc_after.get(
                name,
                {"time_ms": 0, "count": 0}
            )

            delta_time = max(
                after["time_ms"] - before["time_ms"], 0
            )

            delta_count = max(
                after["count"] - before["count"], 0
            )

            gc_time_ms += delta_time
            gc_count += delta_count

            gc_details.append({
                "collector": name,
                "gc_time_ms": delta_time,
                "gc_count": delta_count
            })

        return {
            "rss_start_mb": round(self.rss_start_mb, 2),
            "rss_end_mb": round(self.rss_end_mb, 2),
            "rss_delta_mb": round(
                self.rss_end_mb - self.rss_start_mb, 2
            ),
            "peak_rss_mb": round(self.peak_rss_mb, 2),
            "peak_python_mb": round(self.peak_python_mb, 2),
            "peak_java_mb": round(self.peak_java_mb, 2),
            "jvm_gc_time_ms": gc_time_ms,
            "jvm_gc_count": gc_count,
            "gc_details": gc_details
        }

In [16]:
def build_kg(spark: SparkSession , input_df , normlize : bool = False):

    monitor = ResourceMonitor(
        spark=spark,
        interval=0.2
    )

    monitor.start()

    metrics = None
    result_df = None
    model = None
    config = Config()
    model_config = Model_Embedding()

    required_columns = {
        "document_id",
        "metadata",
        "source_link",
        "title",
        "content"
    }

    missing = required_columns - set(input_df.columns)

    if missing:
        raise ValueError(
            f"Missing columns: {missing}"
        )

    try:
        print("1. Running NLP pipeline...")
        result_df = run_pipeline(spark, input_df)
        result_df.cache()          # cache vì dùng nhiều lần
        result_df.count()

        print("2. Extracting raw vertices & edges...")
        vertices_final, edges_final, occurrences = (
            extract_graph_data(result_df)
        )
        if normlize:
            model = load_embedding_model(model_config)
            if model is not None:
                vertices_final, edges_final = apply_normalization(
                    model_config,
                    spark,
                    result_df,
                    model
                )

        # Ghi bản raw ngay (phòng trường hợp normalize fail)
        save_dataframe(config,occurrences,"occurrences")

        # Save final graph
        save_dataframe(
            config,
            vertices_final,
            "vertices"
        )

        save_dataframe(
            config,
            edges_final,
            "edges"
        )

        print("\n=== Final Stats ===")
        print(f"Vertices: {vertices_final.count()}")
        print(f"Edges   : {edges_final.count()}")

        return vertices_final, edges_final

    finally:
        if "result_df" in locals():
            result_df.unpersist()

        metrics = monitor.stop()

        print("\n========== RESOURCE USAGE ==========")

        print(f"RSS start       : {metrics['rss_start_mb']} MB")
        print(f"RSS end         : {metrics['rss_end_mb']} MB")
        print(f"RSS delta       : {metrics['rss_delta_mb']} MB")
        print(f"Peak RSS        : {metrics['peak_rss_mb']} MB")
        print(f"Peak Python RSS : {metrics['peak_python_mb']} MB")
        print(f"Peak Java RSS   : {metrics['peak_java_mb']} MB")

        print(f"JVM GC time     : {metrics['jvm_gc_time_ms']} ms")
        print(f"JVM GC count    : {metrics['jvm_gc_count']}")

        print("\nGC DETAILS:")

        for item in metrics["gc_details"]:
            print(
                f"{item['collector']}: "
                f"{item['gc_time_ms']} ms, "
                f"{item['gc_count']} collections"
            )

        print("====================================")

In [17]:
print("1. Starting Spark...")
spark = setup_spark(config=Config)
print("Spark:", spark.version)

scala_version = (
    spark.sparkContext._jvm.scala.util.Properties.versionNumberString()
)

print("Scala:", scala_version)

print(
    "JARs:",
    spark.sparkContext._jsc.sc().listJars().toString()
)

1. Starting Spark...
Spark: 3.5.6
Scala: 2.12.18
JARs: Vector(spark://9fc029413a89:33869/jars/com.google.apis_google-api-services-storage-v1-rev20220705-2.0.0.jar, spark://9fc029413a89:33869/jars/io.opencensus_opencensus-contrib-http-util-0.31.1.jar, spark://9fc029413a89:33869/jars/org.conscrypt_conscrypt-openjdk-uber-2.5.2.jar, spark://9fc029413a89:33869/jars/commons-logging_commons-logging-1.1.3.jar, spark://9fc029413a89:33869/jars/com.johnsnowlabs.nlp_jsl-openvino-cpu_2.12-0.1.0.jar, spark://9fc029413a89:33869/jars/com.johnsnowlabs.nlp_spark-nlp_2.12-5.5.0.jar, spark://9fc029413a89:33869/jars/org.apache.datasketches_datasketches-java-6.2.0.jar, spark://9fc029413a89:33869/jars/io.graphframes_graphframes-spark3_2.12-0.12.3.jar, spark://9fc029413a89:33869/jars/com.google.api.grpc_grpc-google-cloud-storage-v2-2.20.1-alpha.jar, spark://9fc029413a89:33869/jars/com.google.http-client_google-http-client-appengine-1.43.0.jar, spark://9fc029413a89:33869/jars/com.amazonaws_aws-java-sdk-kms-1.1

In [18]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

schema = StructType([
    StructField("document_id", StringType(), False),
    StructField("metadata", StringType(), True),
    StructField("source_link", StringType(), True),
    StructField("title", StringType(), True),
    StructField("content", StringType(), False)
])

input_df = spark.createDataFrame(
    sample_list,
    schema=schema
)

input_df.printSchema()
input_df.select(
    "document_id",
    "title",
    "source_link"
).show(5, truncate=False)

root
 |-- document_id: string (nullable = false)
 |-- metadata: string (nullable = true)
 |-- source_link: string (nullable = true)
 |-- title: string (nullable = true)
 |-- content: string (nullable = false)

+-----------+---------------------------------------+-------------------------------------------------------------------------------+
|document_id|title                                  |source_link                                                                    |
+-----------+---------------------------------------+-------------------------------------------------------------------------------+
|64505424   |Tomás Reymão                           |https://en.wikipedia.org/wiki/Tom%C3%A1s%20Reym%C3%A3o                         |
|64513848   |Cheke                                  |https://en.wikipedia.org/wiki/Cheke                                            |
|64512330   |1998 Ball State Cardinals football team|https://en.wikipedia.org/wiki/1998%20Ball%20State%20Cardinals%20foo

## For 1000 input
========== RESOURCE USAGE ==========
RSS start       : 1321.11 MB
RSS end         : 7303.85 MB
RSS delta       : 5982.75 MB
Peak RSS        : 7303.85 MB
Peak Python RSS : 829.19 MB
Peak Java RSS   : 6287.07 MB
JVM GC time     : 18722 ms
JVM GC count    : 81

GC DETAILS:
G1 Young Generation: 18549 ms, 77 collections
G1 Old Generation: 0 ms, 0 collections
G1 Concurrent GC: 173 ms, 4 collections
====================================

In [19]:
vertices, edges = build_kg(spark, input_df , True)

1. Running NLP pipeline...
glove_100d download started this may take some time.
Approximate size to download 145.3 MB
[OK!]
ner_dl download started this may take some time.
Approximate size to download 13.6 MB
[OK!]
2. Extracting raw vertices & edges...
[INFO] Loading model BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
Alexandra Daddario                                 → Cast
 Alexandra Daddario
Jamal Douglas                                      → Cast
 Jamal Douglas
Liana Liberato                                     → Cast
 Liana Liberato
Giorgio Panariello                                 → Cast
Giorgio Panariello
Ramon                                              → Cast
Ramon
San Ramon                                          → Cast
Ramon
Prabhu Mundkur                                     → Cast 
 Prabhu Mundkur
Castelo de Vide                                    → Castelo de Vide's
Castile                                            → Castile
Buildings
Castle Bromwich                                    → Castle Bromwich Aerodrome
Catalina Yachts                                    → Catalina Yachts
Sailboat
United Left Catalonia                              → Catalonia
Federations of United Left
United Left                                        → Ca

In [42]:
"""
vertices = pd.read_parquet(os.path.join(Config.OUTPUT_DIR, "vertices"))
edges = pd.read_parquet(os.path.join(Config.OUTPUT_DIR, "edges"))
occurrences = pd.read_parquet(os.path.join(Config.OUTPUT_DIR, "occurrences"))
print("Vertices:")
print(vertices.head())
print(vertices.info())

print("\nEdges:")
print(edges.head())
print(edges.info())

print("\nOccurrences:")
print(occurrences.head())
print(occurrences.info())
"""

'\nvertices = pd.read_parquet(os.path.join(Config.OUTPUT_DIR, "vertices"))\nedges = pd.read_parquet(os.path.join(Config.OUTPUT_DIR, "edges"))\noccurrences = pd.read_parquet(os.path.join(Config.OUTPUT_DIR, "occurrences"))\nprint("Vertices:")\nprint(vertices.head())\nprint(vertices.info())\n\nprint("\nEdges:")\nprint(edges.head())\nprint(edges.info())\n\nprint("\nOccurrences:")\nprint(occurrences.head())\nprint(occurrences.info())\n'

In [43]:
def validate_graph(
    vertices: DataFrame,
    edges: DataFrame
):
    # 1. Check required columns
    required_v = {"id"}
    required_e = {"src", "dst"}

    assert required_v.issubset(set(vertices.columns))
    assert required_e.issubset(set(edges.columns))

    # 2. Check null IDs
    null_vertices = vertices.filter(
        F.col("id").isNull()
    ).count()

    null_edges = edges.filter(
        F.col("src").isNull() |
        F.col("dst").isNull()
    ).count()

    # 3. Check duplicate vertices
    duplicate_vertices = (
        vertices.groupBy("id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    # 4. Check missing references
    vertex_ids = vertices.select("id").distinct()

    invalid_src = (
        edges.select(F.col("src").alias("id"))
        .join(vertex_ids, on="id", how="left_anti")
        .count()
    )

    invalid_dst = (
        edges.select(F.col("dst").alias("id"))
        .join(vertex_ids, on="id", how="left_anti")
        .count()
    )

    # 5. Check self-loops
    self_loops = edges.filter(
        F.col("src") == F.col("dst")
    ).count()

    report = {
        "null_vertices": null_vertices,
        "null_edges": null_edges,
        "duplicate_vertices": duplicate_vertices,
        "invalid_src": invalid_src,
        "invalid_dst": invalid_dst,
        "self_loops": self_loops
    }

    for key, value in report.items():
        print(f"{key:<25}: {value}")

    valid = all(value == 0 for value in report.values())

    print("\nGraph validation:", "PASSED" if valid else "FAILED")

    return valid

In [23]:
def create_graphframe(
    spark: SparkSession,
    vertices: DataFrame,
    edges: DataFrame
):

    if not validate_graph(vertices, edges):
        raise ValueError("Invalid Knowledge Graph")

    g = GraphFrame(vertices, edges)

    print("GraphFrame initialized successfully")

    print("Vertices:", g.vertices.count())
    print("Edges:", g.edges.count())

    return g

In [24]:
vertices = spark.read.parquet(
    os.path.join(Config.OUTPUT_DIR, "vertices")
)

edges = spark.read.parquet(
    os.path.join(Config.OUTPUT_DIR, "edges")
)

g = create_graphframe(spark, vertices, edges)

null_vertices            : 0
null_edges               : 0
duplicate_vertices       : 0
invalid_src              : 0
invalid_dst              : 0
self_loops               : 0

Graph validation: PASSED
GraphFrame initialized successfully
Vertices: 19935
Edges: 89322


In [25]:
print("=== VERTICES ===")
g.vertices.printSchema()
g.vertices.show(10, truncate=False)

print("=== EDGES ===")
g.edges.printSchema()
g.edges.show(10, truncate=False)

print("Number of vertices:", g.vertices.count())
print("Number of edges:", g.edges.count())

=== VERTICES ===
root
 |-- id: string (nullable = true)
 |-- entity_type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- frequency: long (nullable = true)

+-----------------------------+-----------+-------------------------+---------+
|id                           |entity_type|name                     |frequency|
+-----------------------------+-----------+-------------------------+---------+
|LOC::&c                      |LOC        |&c                       |1        |
|LOC::14september             |LOC        |14September              |1        |
|LOC::[zahn]                  |LOC        |[Zahn]                   |2        |
|LOC::a3013 cove road         |LOC        |A3013 Cove Road          |1        |
|LOC::a323                    |LOC        |A323                     |1        |
|LOC::aargau southern         |LOC        |Aargau Southern          |4        |
|LOC::ab                      |LOC        |Ab                       |2        |
|LOC::aba ngwa north     

In [26]:
from pyspark.sql import functions as F
from graphframes import GraphFrame

edges_forward = g.edges.select(
    "src",
    "dst",
    "relationship"
)

edges_backward = g.edges.select(
    F.col("dst").alias("src"),
    F.col("src").alias("dst"),
    "relationship"
)

edges_bidirectional = (
    edges_forward
    .unionByName(edges_backward)
    .dropDuplicates(["src", "dst"])
)

g_undirected = GraphFrame(
    g.vertices,
    edges_bidirectional
)

print("Vertices:", g_undirected.vertices.count())
print("Bidirectional edges:", g_undirected.edges.count())

Vertices: 19935
Bidirectional edges: 178644


In [27]:
degrees = (
    g_undirected
    .degrees
    .join(g.vertices, "id")
    .select(
        "id",
        "name",
        "entity_type",
        "degree"
    )
    .orderBy(F.desc("degree"))
)

degrees.show(20, truncate=False)

+--------------------------------------+----------------------------------+-----------+------+
|id                                    |name                              |entity_type|degree|
+--------------------------------------+----------------------------------+-----------+------+
|MISC::united states people            |United States\nPeople             |MISC       |1508  |
|LOC::united kingdom                   |United Kingdom                    |LOC        |934   |
|MISC::de france                       |de France                         |MISC       |914   |
|LOC::de france                        |de France                         |LOC        |822   |
|LOC::united states                    |United States                     |LOC        |756   |
|LOC::united states of america         |United States of America          |LOC        |746   |
|MISC::deutschlands                    |Deutschlands                      |MISC       |726   |
|LOC::spain                            |Spain     

In [30]:
OUTPUT = "kg_output/gephi"

(
    g_undirected.vertices
    .select("id", "name", "entity_type")
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(f"{OUTPUT}/nodes")
)

(
    g_undirected.edges
    .select("src", "dst")
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(f"{OUTPUT}/edges")
)

In [31]:
pr_graph = g_undirected.pageRank(
    resetProbability=0.15,
    maxIter=10
)

pagerank = (
    pr_graph.vertices
    .select(
        "id",
        "name",
        "entity_type",
        "pagerank"
    )
    .orderBy(F.desc("pagerank"))
)

pagerank.show(20, truncate=False)

+--------------------------------------+----------------------------------+-----------+------------------+
|id                                    |name                              |entity_type|pagerank          |
+--------------------------------------+----------------------------------+-----------+------------------+
|MISC::united states people            |United States\nPeople             |MISC       |74.87613586027243 |
|LOC::united kingdom                   |United Kingdom                    |LOC        |45.818968323741366|
|MISC::de france                       |de France                         |MISC       |44.97987509130858 |
|MISC::deutschlands                    |Deutschlands                      |MISC       |43.015383921643526|
|LOC::de france                        |de France                         |LOC        |37.194077822450375|
|LOC::united states                    |United States                     |LOC        |36.90863778415568 |
|LOC::united states of america       

In [32]:
num_vertices = g.vertices.count()
num_edges = g.edges.count()

avg_degree = (
    degrees
    .agg(F.avg("degree").alias("avg_degree"))
    .first()["avg_degree"]
)

max_degree = (
    degrees
    .agg(F.max("degree").alias("max_degree"))
    .first()["max_degree"]
)

print("========== GRAPH SUMMARY ==========")
print("Number of vertices:", num_vertices)
print("Number of edges:", num_edges)
print("Average degree:", avg_degree)
print("Maximum degree:", max_degree)
print("===================================")

========== GRAPH SUMMARY ==========
Number of vertices: 19935
Number of edges: 89322
Average degree: 19.261846999838266
Maximum degree: 1508
